In [2]:
from agents import Agent

# To do list:

* Input: Start date, End date, short/medium/long
* From Start date, End date => List all range


In [23]:
import os, json
import pandas as pd
import numpy as np
import mysql.connector
from dotenv import load_dotenv
from typing import List, Union, Optional

load_dotenv()

# ----------------------------------------------------------------------
# DB Connection
# ----------------------------------------------------------------------
def get_connection():
    """Efficient MySQL connection"""
    return mysql.connector.connect(
        host=os.getenv("MYSQL_HOST"),
        user=os.getenv("MYSQL_USER"),
        password=os.getenv("MYSQL_PASSWORD"),
        database=os.getenv("MYSQL_DATABASE"),
        connection_timeout=5
    )

# ----------------------------------------------------------------------
# Main Function
# ----------------------------------------------------------------------
def get_latest_price(
    coin_symbol: Union[List[str], str],
    time_horizons: Union[List[str], str],
    start_date: Optional[str] = None,
    end_date: Optional[str] = None
) -> str:
    """
    Keep full kline data but compress funding rates with sparse timestamps and summary stats.
    Open interest remains 4h-aggregated for alignment.
    """
    if isinstance(coin_symbol, str):
        coin_symbol = [coin_symbol]
    if isinstance(time_horizons, str):
        time_horizons = [time_horizons]

    conn = get_connection()
    cursor = conn.cursor(dictionary=True)
    results = {}

    def run_query(query: str, params: tuple = ()):
        cursor.execute(query, params)
        rows = cursor.fetchall()
        return pd.DataFrame(rows) if rows else pd.DataFrame()

    # --- Date filter ---
    date_filter = ""
    date_params = []
    if start_date:
        date_filter += " AND DATE(open_time) >= %s"
        date_params.append(start_date)
    if end_date:
        date_filter += " AND DATE(open_time) <= %s"
        date_params.append(end_date)

    # ------------------------------------------------------------------
    for sym in coin_symbol:
        sym = sym.lower().strip()
        sym_results = {}

        for horizon in time_horizons:
            horizon = horizon.lower().strip()
            dfs = {}

            if horizon == "short":
                # --- 4h kline (keep all OHLCV) ---
                dfs["kline"] = run_query(
                    f"""
                    SELECT close_time, open_price, high_price, low_price, close_price, volume
                    FROM finance_services.crypto_kline_hours
                    WHERE `interval`='4h' AND symbol=%s {date_filter}
                    ORDER BY close_time
                    """,
                    (sym, *date_params)
                )

                # --- Funding rates (hybrid compression) ---
                df_fund = run_query(
                    f"""
                    SELECT funding_time, funding_rate
                    FROM finance_services.futures_funding_rates
                    WHERE symbol=%s
                    {" AND DATE(funding_time) BETWEEN %s AND %s" if start_date and end_date else ""}
                    ORDER BY funding_time
                    """,
                    (sym, *(date_params if len(date_params) == 2 else []))
                )

                if not df_fund.empty:
                    df_fund = df_fund.sort_values("funding_time").reset_index(drop=True)
                    df_fund["funding_delta"] = df_fund["funding_rate"].diff().fillna(0) * 1e4
                    base_rate = float(df_fund["funding_rate"].iloc[0])
                    deltas = df_fund["funding_delta"].iloc[1:]
                    dfs["funding"] = {
                        "base_rate": base_rate,
                        "stats": {
                            "avg_delta": round(float(deltas.mean()), 3),
                            "max_delta": round(float(deltas.max()), 3),
                            "min_delta": round(float(deltas.min()), 3),
                            "volatility": round(float(deltas.std()), 3)
                        },
                        "series": df_fund.iloc[::max(1, len(df_fund)//8)][["funding_time","funding_rate"]]
                                  .rename(columns={"funding_time": "ts", "funding_rate": "rate"})
                                  .round(8)
                                  .to_dict(orient="records"),
                        "count": len(df_fund)
                    }
                else:
                    dfs["funding"] = {}

                # --- Open Interest (4h aggregation) ---
                dfs["open_interest"] = run_query(
                    f"""
                    SELECT 
                        DATE_FORMAT(MIN(timestamp),'%Y-%m-%d %H:00:00') AS ts,
                        AVG(open_interest_usd) AS open_interest_usd,
                        AVG(open_interest_coin) AS open_interest_coin
                    FROM finance_services.futures_open_interests
                    WHERE symbol=%s
                    {" AND DATE(timestamp) BETWEEN %s AND %s" if start_date and end_date else ""}
                    GROUP BY FLOOR(UNIX_TIMESTAMP(timestamp)/(4*3600))
                    ORDER BY ts
                    """,
                    (sym, *(date_params if len(date_params) == 2 else []))
                )

            else:
                continue

            # --- Clean and summarize ---
            summary = []
            for name, df in dfs.items():
                if isinstance(df, pd.DataFrame):
                    if "volume" in df.columns:
                        df = df[df["volume"] > 0].copy()
                    float_cols = df.select_dtypes(include="float").columns
                    df.loc[:, float_cols] = df[float_cols].round(3)
                    time_col = next((c for c in ["open_time", "close_time", "funding_time", "timestamp", "ts"] if c in df.columns), None)
                    tmin = df[time_col].min() if time_col else None
                    tmax = df[time_col].max() if time_col else None
                    summary.append({
                        "dataset": name,
                        "rows": len(df),
                        "columns": list(df.columns),
                        "start_time": tmin,
                        "end_time": tmax
                    })
                    dfs[name] = df
                else:
                    summary.append({
                        "dataset": name,
                        "rows": len(df.get("series", [])),
                        "columns": list(df.keys()),
                        "start_time": df["series"][0]["ts"] if df.get("series") else None,
                        "end_time": df["series"][-1]["ts"] if df.get("series") else None
                    })

            sym_results[horizon] = {
                "data": {k: (v.to_dict(orient="records") if isinstance(v, pd.DataFrame) else v) for k, v in dfs.items()},
                "summary": summary
            }

        results[sym] = sym_results

    cursor.close()
    conn.close()
    return json.dumps(results, default=str, separators=(",", ":"))


In [24]:
get_latest_price('btc', 'short', '2025-10-01', '2025-10-08' )

'{"btc":{"short":{"data":{"kline":[{"close_time":"2025-10-01 07:00:00","open_price":114359.99,"high_price":114754.87,"low_price":113765.42,"close_price":114048.93,"volume":1977.488},{"close_time":"2025-10-01 11:00:00","open_price":114048.94,"high_price":114551.76,"low_price":113966.67,"close_price":114176.92,"volume":2042.571},{"close_time":"2025-10-01 15:00:00","open_price":114176.93,"high_price":114740.0,"low_price":114151.0,"close_price":114539.02,"volume":1996.434},{"close_time":"2025-10-01 19:00:00","open_price":114539.02,"high_price":116795.85,"low_price":114484.43,"close_price":116789.58,"volume":5217.93},{"close_time":"2025-10-01 23:00:00","open_price":116789.57,"high_price":117649.99,"low_price":116363.38,"close_price":117423.73,"volume":4518.346},{"close_time":"2025-10-02 03:00:00","open_price":117423.73,"high_price":118199.0,"low_price":116724.56,"close_price":117439.53,"volume":3814.323},{"close_time":"2025-10-02 07:00:00","open_price":117439.53,"high_price":118649.1,"low_p

In [34]:
from datetime import datetime, timedelta

class DateRangeSplitter:
    def __init__(self, start_date: str, end_date: str, chunk_days: int):
        self.start_date = datetime.strptime(start_date, "%Y-%m-%d")
        self.end_date = datetime.strptime(end_date, "%Y-%m-%d")
        self.chunk_days = chunk_days
        self._ranges = self._generate_ranges()

    def _generate_ranges(self):
        """Internal: generate list of date range dicts."""
        ranges = []
        current = self.start_date
        while current < self.end_date:
            next_end = current + timedelta(days=self.chunk_days)
            if next_end > self.end_date:
                next_end = self.end_date
            ranges.append({
                "start": current.strftime("%Y-%m-%d"),
                "end": next_end.strftime("%Y-%m-%d"),
            })
            current = next_end
        return ranges

    # --- Public methods ---
    def get_meta(self):
        """Return metadata about the split."""
        return {
            "start_date": self.start_date.strftime("%Y-%m-%d"),
            "end_date": self.end_date.strftime("%Y-%m-%d"),
            "chunk_days": self.chunk_days,
            "total_days": (self.end_date - self.start_date).days,
            "chunk_count": len(self._ranges)
        }

    def get_ranges(self):
        """Return the full detailed list of ranges."""
        return self._ranges

    def get_array(self):
        """Return only [start, end] pairs."""
        return [[r["start"], r["end"]] for r in self._ranges]

    def __repr__(self):
        return f"<DateRangeSplitter chunks={len(self._ranges)} days_per_chunk={self.chunk_days}>"


In [35]:
splitter = DateRangeSplitter("2025-06-01", "2025-11-03", 3)

print("Meta:")
print(splitter.get_meta())

print("\nDetailed ranges:")
for r in splitter.get_ranges():
    print(r)

print("\nArray form:")
print(splitter.get_array())


Meta:
{'start_date': '2025-06-01', 'end_date': '2025-11-03', 'chunk_days': 3, 'total_days': 155, 'chunk_count': 52}

Detailed ranges:
{'start': '2025-06-01', 'end': '2025-06-04'}
{'start': '2025-06-04', 'end': '2025-06-07'}
{'start': '2025-06-07', 'end': '2025-06-10'}
{'start': '2025-06-10', 'end': '2025-06-13'}
{'start': '2025-06-13', 'end': '2025-06-16'}
{'start': '2025-06-16', 'end': '2025-06-19'}
{'start': '2025-06-19', 'end': '2025-06-22'}
{'start': '2025-06-22', 'end': '2025-06-25'}
{'start': '2025-06-25', 'end': '2025-06-28'}
{'start': '2025-06-28', 'end': '2025-07-01'}
{'start': '2025-07-01', 'end': '2025-07-04'}
{'start': '2025-07-04', 'end': '2025-07-07'}
{'start': '2025-07-07', 'end': '2025-07-10'}
{'start': '2025-07-10', 'end': '2025-07-13'}
{'start': '2025-07-13', 'end': '2025-07-16'}
{'start': '2025-07-16', 'end': '2025-07-19'}
{'start': '2025-07-19', 'end': '2025-07-22'}
{'start': '2025-07-22', 'end': '2025-07-25'}
{'start': '2025-07-25', 'end': '2025-07-28'}
{'start': '

In [ ]:
# Run the AI workflow:
from agents import Agent, Runner
from pydantic import BaseModel

class TechnicalAnalysis(BaseModel):
    start_date: str
    end_date: str
    analysis: str

class ReviewerAnalysis(BaseModel):
    start_date: str
    end_date: str
    analysis: str
    review: str

report : List[TechnicalAnalysis] = []
reviewer_report : List[ReviewerAnalysis] = []
count = 0 

for r in splitter.get_ranges():
    # Get the data for the current range
    output = get_latest_price('btc', 'short', r['start'], r['end'])

    # Construct the prompt for the AI
    system_prompt = """Pass"""
    user_prompt = """Hello"""

    # Run the AI
    agent = Agent(
        name = "technical-analysis-agent",
        model = "gpt-5-mini",
        instructions = system_prompt
    )
    result = await Runner.run(agent, user_prompt)
    print(result.final_output)
    technical_analysis = TechnicalAnalysis(
        analysis = result.final_output,
        start_date = r['start'],
        end_date = r['end']
    )

    # Send the technical analysis to the reviewer
    reviewer_prompt = f"""
    Review the following technical analysis:
    {json.dumps(technical_analysis.model_dump(), indent=2)}
    """
    print(reviewer_prompt)


    # 
    
    
    # Save the technical analysis to the report
    report.append(technical_analysis)


    # Break Loops for testing
    count += 1
    if count > 1:
        break # Break after the first iteration for testing
    

Hi — how can I help you today?
Hi — how can I help you today?
Hi — how can I help you today?


In [57]:
# Save the report to a json file

with open('report.json', 'w', encoding='utf-8') as f:
    json.dump([r.model_dump() for r in report], f, ensure_ascii=False, indent=2)


In [ ]:
# Define the reviewer AI
# from agents import WebSearchTool, CodeInterpreterTool
reviewer = Agent(
    name = "reviewer-agent",
    model = "gpt-5-mini",
    instructions = """You are a technical analyst reviewing the technical analysis report."""
)



